# Limpeza da base CPGF (Cartão de Pagamento do Governo Federal)

Fonte: Portal da Transparência. O arquivo vem com `sep=';'` e encoding `latin-1`.

In [ ]:
import pandas as pd

# 1. Carregando a base de dados
caminho_dados = '../202608_CPGF.csv'
df = pd.read_csv(caminho_dados, sep=';', encoding='latin-1', dtype=str)

print(f"Linhas e colunas originais: {df.shape}")
df.head()

In [ ]:
# 2. Selecionando apenas as colunas úteis (redução da base)
# Nesta versão do arquivo, fornecedor/pagamento se chamam FAVORECIDO/TRANSAÇÃO
colunas_uteis = [
    'NOME ÓRGÃO SUPERIOR',
    'NOME PORTADOR',
    'NOME FAVORECIDO',
    'TRANSAÇÃO',
    'DATA TRANSAÇÃO',
    'VALOR TRANSAÇÃO',
]

df_limpo = df[colunas_uteis].copy()
df_limpo.head()

In [ ]:
# 3. Tratando os valores financeiros: "1.500,00" -> 1500.00
df_limpo['VALOR TRANSAÇÃO'] = (
    df_limpo['VALOR TRANSAÇÃO']
    .str.replace('.', '', regex=False)   # tira o ponto de milhar
    .str.replace(',', '.', regex=False)  # troca vírgula por ponto
    .astype(float)
)

df_limpo['VALOR TRANSAÇÃO'].describe()

In [ ]:
# 4. Tratando dados nulos e datas
# Linhas "Informações protegidas por sigilo" não têm data e são removidas aqui.
# Antes de remover, medimos o impacto para registrar na documentação.
print(df_limpo.isna().sum())

mascara_nulos = df_limpo.isna().any(axis=1)
valor_total = df_limpo['VALOR TRANSAÇÃO'].sum()
valor_removido = df_limpo.loc[mascara_nulos, 'VALOR TRANSAÇÃO'].sum()
print(f"\nLinhas removidas: {mascara_nulos.sum()} ({mascara_nulos.mean():.1%} das linhas)")
print(f"Valor removido: R$ {valor_removido:,.2f} ({valor_removido / valor_total:.1%} do valor total)")

df_limpo = df_limpo.dropna()
df_limpo['DATA TRANSAÇÃO'] = pd.to_datetime(df_limpo['DATA TRANSAÇÃO'], format='%d/%m/%Y')

print(f"\nLinhas e colunas após limpeza: {df_limpo.shape}")
df_limpo.dtypes

In [ ]:
# 5. Tratando o falso ausente "NAO SE APLICA" no favorecido
# Em saques não existe favorecido; o governo preenche com "NAO SE APLICA".
# Trocamos por um rótulo explícito para não virar o "fornecedor" mais frequente.
mascara_nsa = df_limpo['NOME FAVORECIDO'] == 'NAO SE APLICA'
print("Transações com 'NAO SE APLICA':")
print(df_limpo.loc[mascara_nsa, 'TRANSAÇÃO'].value_counts())

df_limpo.loc[mascara_nsa, 'NOME FAVORECIDO'] = 'SAQUE (SEM FAVORECIDO)'
print(f"\nLinhas ajustadas: {mascara_nsa.sum()}")

In [ ]:
# 6. Verificando duplicatas (apenas diagnóstico, sem remoção)
# O CSV não tem ID de transação; linhas iguais costumam ser compras repetidas legítimas
# (mesmo portador, dia, favorecido e valor). Por isso as linhas são mantidas.
duplicadas = df_limpo.duplicated(keep=False)
print(f"Linhas repetidas (excedentes): {df_limpo.duplicated().sum()}")
df_limpo[duplicadas].sort_values(list(df_limpo.columns)).head(6)

In [ ]:
# 7. Padronizando nomes de colunas (snake_case, sem acentos e espaços)
df_limpo = df_limpo.rename(columns={
    'NOME ÓRGÃO SUPERIOR': 'orgao_superior',
    'NOME PORTADOR': 'portador',
    'NOME FAVORECIDO': 'favorecido',
    'TRANSAÇÃO': 'tipo_transacao',
    'DATA TRANSAÇÃO': 'data',
    'VALOR TRANSAÇÃO': 'valor',
})
df_limpo = df_limpo.reset_index(drop=True)

# Checagens finais da base
assert df_limpo.isna().sum().sum() == 0, "Ainda existem nulos"
assert (df_limpo['valor'] > 0).all(), "Existem valores não positivos"
assert not (df_limpo['favorecido'] == 'NAO SE APLICA').any()

print(f"Base final: {df_limpo.shape}")
df_limpo.info()

In [ ]:
# 8. Amostra da base final tratada
df_limpo.head(10)

In [ ]:
# 9. Exportando a base final tratada para o grupo usar
caminho_saida = '../base_cpgf_tratada.csv'
df_limpo.to_csv(caminho_saida, index=False)

print(f"Base exportada com sucesso em {caminho_saida}")

## Origem dos dados

Os dados vêm do **Portal da Transparência do Governo Federal**, conjunto *Cartão de Pagamento do Governo Federal (CPGF)*, extrato de **agosto de 2026** (arquivo `202608_CPGF.csv`). Cada linha é uma transação feita com o cartão corporativo por um servidor (portador) vinculado a um órgão federal. As transações do extrato ocorreram entre 26/06/2026 e 29/07/2026.

O arquivo original tem 15.237 linhas e 15 colunas, separador `;`, codificação `latin-1`, valores monetários no padrão brasileiro (`1.234,56`) e datas no formato `dd/mm/aaaa`. Os CPFs dos portadores vêm parcialmente mascarados pelo próprio portal.

## Processo de ETL

1. **Extração:** leitura do CSV com `sep=';'`, `encoding='latin-1'` e todas as colunas como texto, para evitar conversões automáticas erradas.
2. **Redução:** das 15 colunas, ficaram 6. Códigos numéricos (órgão, unidade gestora), CPF/CNPJ e ano/mês do extrato foram descartados por não agregarem à análise.
3. **Tratamento monetário:** remoção do ponto de milhar, troca da vírgula decimal por ponto e conversão para `float`.
4. **Tratamento de nulos:** foram removidas 3.926 linhas (25,8%) do tipo *"Informações protegidas por sigilo"*, que não têm data nem CPF. **Atenção:** essas linhas somam R$ 4.532.498,68, ou **38,7% do valor total** do extrato (R$ 11.724.649,65). Totais de gasto calculados na base tratada, especialmente por órgão, ficam subestimados.
5. **Tratamento de datas:** conversão de `dd/mm/aaaa` para `datetime`.
6. **Falso ausente:** em todos os 1.272 saques o favorecido vinha como `"NAO SE APLICA"`. O texto foi trocado por `"SAQUE (SEM FAVORECIDO)"` para não ser confundido com um fornecedor real.
7. **Duplicatas:** há 476 linhas idênticas a outras. Como o arquivo não tem identificador de transação, elas foram **mantidas**, por serem provavelmente compras repetidas legítimas.
8. **Padronização:** colunas renomeadas para `snake_case`, sem acentos.
9. **Carga:** exportação para `data/base_cpgf_tratada.csv` (UTF-8, separador `,`, sem coluna de índice).

**Resultado:** 11.311 linhas e 6 colunas, sem nulos e com valores entre R$ 0,10 e R$ 23.556,00.

## Dicionário de dados (`base_cpgf_tratada.csv`)

| Coluna | Tipo | Coluna original | Descrição | Exemplo |
|---|---|---|---|---|
| `orgao_superior` | texto | NOME ÓRGÃO SUPERIOR | Órgão superior (ministério ou equivalente) ao qual o portador está vinculado. 29 valores distintos. | Advocacia-Geral da União |
| `portador` | texto | NOME PORTADOR | Nome do servidor titular do cartão que fez a transação. | ISABELA RODRIGUES TEIXEIRA |
| `favorecido` | texto | NOME FAVORECIDO | Estabelecimento/fornecedor que recebeu o pagamento. Em saques, `SAQUE (SEM FAVORECIDO)`. | VLB COMERCIO VAREJISTA DE FERRAGENS LTDA |
| `tipo_transacao` | texto (categoria) | TRANSAÇÃO | Tipo da operação: compra à vista ou saque, em R$ ou moeda internacional (INT$). | COMPRA A/V - R$ - APRES |
| `data` | data (`aaaa-mm-dd`) | DATA TRANSAÇÃO | Data em que a transação ocorreu. | 2026-07-13 |
| `valor` | decimal (R$) | VALOR TRANSAÇÃO | Valor da transação em reais, separador decimal `.`. | 97.49 |